# Trade Agreements, Vaccine Prices & Immunization Coverage: Causal Mediation Analysis

**Causal structure:**
- **T** (Treatment): Trade liberalization score
- **M** (Mediator): Average vaccine price (WHO region × income group × GAVI status)
- **Y** (Outcome): Immunization coverage rate (% of target population vaccinated)
- **W** (Confounders): GDP per capita, health expenditure, government effectiveness, partner-aggregate provision scores


In [ ]:
# Run once, then restart the kernel before running the imports cell
!pip install numpy==1.26.4 --force-reinstall -q
!pip install pandas country_converter matplotlib seaborn dowhy econml xgboost shap openpyxl xlrd -q


In [ ]:
import os
import pandas as pd
import country_converter as coco
from itertools import permutations
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns
import numpy as np
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')


## 1. Data Loading

### 1.1 Pharma Preferential Tariffs (HS2 = 30)

In [ ]:
PHARMA_PATH = (
    r'C:\Users\srima\OneDrive\Desktop\INSY674\Individual Project 1'
    r'\1.-preferential-tariffs.csv (1)\Chemicals_Allied_Industries.csv'
)

pharma_df = pd.read_csv(PHARMA_PATH)
pharma_df = pharma_df.rename(columns={'mfn': 'standard_rate', 'prf': 'preferential_rate'})
pharma_df = pharma_df[pharma_df['hs2'] == 30].copy()

print(f'Shape: {pharma_df.shape}')
print(f'Columns: {pharma_df.columns.tolist()}')
print(f'Missing values:\n{pharma_df.isnull().sum()}')
pharma_df.head()


### 1.2 WHO MI4A Vaccine Prices

In [ ]:
VACCINE_PATH = (
    r'C:\Users\srima\OneDrive\Desktop\INSY674\Individual Project 1'
    r'\who-mi4a-dataset-final-september-2025.xlsx'
)

vaccine_df = pd.read_excel(VACCINE_PATH, sheet_name='Vaccine purchase database')

print(f'Shape: {vaccine_df.shape}')
print(f'Columns: {vaccine_df.columns.tolist()}')
print(f'Missing values:\n{vaccine_df.isnull().sum()}')
vaccine_df.head()


### 1.3 WUENIC Immunization Coverage

In [ ]:
COVERAGE_PATH = r'C:\Users\srima\Downloads\wuenic2024rev_web-update.xlsx'

xl = pd.ExcelFile(COVERAGE_PATH)
print(f'Sheet names: {xl.sheet_names}')


In [ ]:
# Update sheet_name below if needed after inspecting the output above
coverage_df = pd.read_excel(COVERAGE_PATH, sheet_name=xl.sheet_names[0])

print(f'Shape: {coverage_df.shape}')
print(f'Columns: {coverage_df.columns.tolist()}')
print(f'Sample values (first col): {coverage_df.iloc[:, 0].dropna().unique()[:10]}')
print(f'Missing values:\n{coverage_df.isnull().sum()}')
coverage_df.head()
